# 攻防箭 · 可播放的比赛动画

23×12 米，75×75 厘米单靶架、五个独立靶点。攻方在 120 秒内清空五孔获胜。使用普通 CPU，无需 GPU、Pygame 窗口或视频编码器。选择“运行全部”，计算结束后点击播放。所有几何和能力参数仍为待校准假设。


In [1]:
from pathlib import Path
import subprocess, sys
PROJECT_DIR = Path("/content/archery-tag-simulate")
REPO_URL = "https://github.com/Echoudexigaigu/archery-tag-simulate.git"
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    if not (PROJECT_DIR / ".git").exists():
        raise RuntimeError("项目目录没有 Git 历史，请先备份，再使用一个新的 Colab 运行时。")
    remote = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "remote", "get-url", "origin"], text=True).strip()
    if remote.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("已有目录属于其他仓库；请使用新的运行时。")
    dirty = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True).strip()
    if dirty:
        print("保留已有未提交改动，不自动更新。请先保存，再决定如何更新。")
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if not (PROJECT_DIR / "src/archery_tag_balance/environment.py").is_file():
    raise RuntimeError("已有目录还是旧骨架。保存改动后更新仓库，或在新运行时重新运行。")
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True).strip())

if "archery_tag_balance" in sys.modules:
    print("当前运行时已导入过仿真模块；若仓库有更新，请先重启会话，以加载最新版本。")
if not (PROJECT_DIR / "src/archery_tag_balance/replay.py").is_file():
    raise RuntimeError("仓库未更新到动画版本；请保存本地改动后更新，或使用新运行时。")


901615eb08fdabb27f97202889baebd800c0a416


## 选择阵容、靶架坐标和随机种子

每队 veterans 是老手人数，个体 η 由配置范围抽样。奇数种子会镜像 y 轴，减小上下站位偏差；动画显示实际镜像后的布局。改变参数后从本节重新运行。


In [2]:
import json
from archery_tag_balance import SimulationEngine
from archery_tag_balance.experiments import save_json, source_digest
config = json.loads((PROJECT_DIR / "configs/attack_defense_23x12.json").read_text())
SEED = 1000
config["rack"]["position_xyz_m"] = [21.5, 6.0, 1.1]
config["teams"]["attack"]["veterans"] = 2
config["teams"]["defense"]["veterans"] = 2
# 可尝试靶架前移到 [18.0, 6.0, 1.1]，或改成老手 3 对 1。
summary, env = SimulationEngine(config).episode(
    seed=SEED, record=True, record_interval_s=0.1, check_invariants=True
)
print(f"胜方: {summary['winner']}；比赛时长: {summary['duration_s']:.1f} 秒；记录: {len(env.trace)} 帧")
run_dir = PROJECT_DIR / "runs" / f"animation_{SEED}"
save_json(run_dir / "config.json", config)
save_json(run_dir / "episode.json", {"summary": summary, "trace": env.trace, "events": env.events, "source_sha256": source_digest()})


胜方: attack；比赛时长: 46.4 秒；记录: 465 帧


## 播放比赛动画

先运行上一格的单局仿真，再点播放。支持暂停、逐帧、拖动和倍速。黄色箭头为飞行箭的俯视投影，右边是同一块靶架的五孔正视图。记录约每 0.1 秒一帧，浏览器只负责回放，不影响仿真决策。HTML 包含比赛轨迹，可下载后离线播放。


In [3]:
from IPython.display import HTML, display
from archery_tag_balance.replay import replay_html
html = replay_html(config, env, speed=2.0)
display(HTML(html))
replay_path = run_dir / "replay.html"
replay_path.write_text(html, encoding="utf-8")
print("回放已保存:", replay_path)


回放已保存: /content/archery-tag-simulate/runs/animation_1000/replay.html


## 可选：下载回放

运行下一格下载 HTML，可在浏览器离线打开。Colab 输出在重新连接后可能需要重新运行生成；如不显示控件，请重新运行播放单元格。运行时文件不永久保存，需下载或自行备份到 Drive。回放使用全局观察者视角，不代表玩家能看见被掩体遮挡的敌人。


下载时在新的代码格执行：
```python
from google.colab import files
files.download(str(replay_path))
```


In [4]:
from google.colab import files
files.download(str(replay_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>